# OneRef sur Colab
Runtime recommandé : **GPU L4** (bf16). Exécute les cellules dans l'ordre ; les sorties sont sur Drive donc un run coupé **reprend** là où il s'est arrêté.

In [ ]:
# 1. GPU + profil vLLM (modeles 4B: Qwen3-VL-4B, InternVL3.5-4B, Gemma 4 E2B)
import subprocess, json, os
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],capture_output=True,text=True).stdout.strip()
print(gpu)
T4 = "T4" in gpu
vllm_args = ["--max-model-len","8192","--gpu-memory-utilization","0.85","--trust-remote-code","--enable-auto-tool-choice","--tool-call-parser","hermes"]
if T4:
    vllm_args += ["--dtype","half","--enforce-eager"]   # pas de bf16 sur T4 ; eager = moins de memoire
os.environ["ONEREF_VLLM_ARGS"] = json.dumps(vllm_args)


In [ ]:
# 2. Drive (donnees/sorties) + code depuis GitHub (branche Test_implementation)
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/OneRef"
os.makedirs(DRIVE, exist_ok=True)
!rm -rf /content/OneRef && git clone -q --depth 1 -b Test_implementation https://github.com/RVilchezzz/OneRef.git /content/OneRef
!rm -rf /content/OneRef_Colab && cp -r /content/OneRef/Colab /content/OneRef_Colab && cp -r /content/OneRef/configs /content/OneRef_Colab/configs
%cd /content/OneRef_Colab


In [ ]:
# 3. Installation (vLLM peut remplacer torch : si Colab le demande, redémarre le runtime puis relance depuis la cellule 1)
!pip install -q -r requirements_colab.txt

In [ ]:
# 4. Hugging Face (secret Colab "HF_TOKEN" ; accepter la licence DINOv3 sur le Hub)
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [ ]:
# 5. Variables d'environnement du projet
os.environ.update({
    "ONEREF_MMAD_ROOT": f"{DRIVE}/MMAD",
    "ONEREF_MMAD_JSON": f"{DRIVE}/MMAD/mmad.json",
    "ONEREF_OUTPUTS_DIR": f"{DRIVE}/outputs",
    "ONEREF_SPLITS_DIR": f"{DRIVE}/splits",
    "ONEREF_LLM_PROVIDER": "vllm",          # le code par défaut utilise un API hébergée ; ici : vLLM local, un serveur par passe
    "ONEREF_LLM_BASE_URL": "http://localhost:8000/v1",
    "ONEREF_LLM_API_KEY": "EMPTY",
    "ONEREF_VLLM_AUTOSTART": "true",
    "ONEREF_DEVICE": "cuda",
    "PYTHONPATH": "/content/OneRef_Colab",
})
# MMAD : "subset" (N images du split dev + leurs references, ~quelques Mo : recommande pour tester) | "sample" (zip DS-MVTec ; ses
# references sont dans MVTec-AD.zip donc --available-only ne trouve rien) | "full" (~28 Go) | "none" (deja sur Drive)
MMAD_MODE = "subset"
N_SAMPLE = 50
SAMPLE_ZIPS = ["DS-MVTec.zip"]   # DS-MVTec 1.6 Go, VisA 1.9 Go, MVTec-AD 5.3 Go, MVTec-LOCO 6.1 Go, GoodsAD 13.3 Go
OPT = "--available-only" if MMAD_MODE == "sample" else (f"--sample {N_SAMPLE}" if MMAD_MODE == "subset" else "")   # n'utiliser que les images téléchargées

# Config 4B (cloné avec le code) : modeles, judge, calibration de l'expert
import shlex
for line in open("configs/2026-11-19_4B_colab_vllm.env", encoding="utf-8"):
    line = line.strip()
    if line and not line.startswith("#"):
        k, v = line.split("=", 1); os.environ[k] = shlex.split(v)[0] if v else ""


In [ ]:
# 5b. Telecharger MMAD sur Drive (UNE seule fois). "subset" : mmad.json + N images dev et leurs references ; "sample" : un zip ; "full" : ~28 Go.
# Sauter cette cellule si MMAD est deja dans MyDrive/OneRef/MMAD (MMAD_MODE = "none").
if MMAD_MODE == "subset":
    from download_mmad import download_mmad
    download_mmad(f"{DRIVE}/MMAD", [])   # mmad.json seul
    os.makedirs(f"{DRIVE}/splits", exist_ok=True)
    !python /content/OneRef/scripts/fetch_mmad_subset.py --split dev --sample {N_SAMPLE} --root {DRIVE}/MMAD --splits-dir {DRIVE}/splits
elif MMAD_MODE in ("sample", "full"):
    from download_mmad import ALL_ZIPS, download_mmad
    download_mmad(f"{DRIVE}/MMAD", ALL_ZIPS if MMAD_MODE == "full" else SAMPLE_ZIPS)
else:
    print("MMAD_MODE = none : rien a telecharger")


In [ ]:
# 6. Smoke test sans GPU
!python -m pytest -q
!python -m inspector.main --split dev --mock --limit 20 --run-name colab_smoke {OPT}


In [ ]:
# 7. Sonde de cout : N_SAMPLE images, vrais modeles (mesure le temps et les unites Colab)
!python -m inspector.main --split dev --run-name probe {OPT}


In [ ]:
# 8. Run dev complet (reprenable : relance la cellule après une coupure)
!python -m inspector.main --split dev --run-name dev01 {OPT}

In [ ]:
# 9. Dernières lignes du log JSONL
!tail -n 3 "$DRIVE"/outputs/dev01/*.jsonl | cut -c1-600

## Test (une seule fois, config figée)
Ne lance **jamais** le split test pour régler des paramètres.

In [ ]:
# !python -m inspector.main --split test --frozen-config --run-name test01 {OPT}